In [21]:
from netgen.meshing import Mesh as NGMesh, MeshPoint, Element1D, Element0D, Pnt
from ngsolve import *
from netgen.occ import *
from ngsolve.webgui import Draw, FieldLines, AddFieldLines

import matplotlib.pylab as plt
import numpy as np

In [22]:
def CapacitorGeometry():

    air = MoveTo(0, 0).RectangleC(30, 30).Face()
    air.edges.name = "Outer"
    air.faces.name = "air"

    el_u = MoveTo(0, 1).RectangleC(5, 0.5).Face()
    el_u.edges.name = "el_u"
    el_u.faces.name = "el_u"

    el_d = MoveTo(0, -1).RectangleC(5, 0.5).Face()
    el_d.edges.name = "el_d"
    el_d.faces.name = "el_d"

    dielectric = MoveTo(0, 0).RectangleC(4, 1.5).Face()
    dielectric.faces.name = "dielectric"

    shape = Glue([air - dielectric, dielectric])
    shape = shape - el_u - el_d

    shape.edges["el.*"].maxh=0.2
    shape.vertices["el.*"].maxh=0.2
    
    return shape


def CapacitorMesh(shape, h_max):
    
    mesh = Mesh(OCCGeometry(shape, dim=2).GenerateMesh(maxh=h_max))

    return mesh


def CapacitorSolver(mesh, FE_order, epsr):

    fes = H1(mesh, order=FE_order, dirichlet="el.*")

    u = fes.TrialFunction()
    v = fes.TestFunction()

    gfu = GridFunction(fes)
    gfu.Interpolate(mesh.BoundaryCF({"el_u":1, "el_d":-1 }), mesh.Boundaries(".*"))

    a = BilinearForm(epsr*grad(u)*grad(v)*dx).Assemble()
    
    inv = a.mat.Inverse(freedofs=fes.FreeDofs())
    gfu.vec.data -= inv@a.mat * gfu.vec

    return gfu

In [23]:
h_max = 1
geo = CapacitorGeometry()
mesh = CapacitorMesh(geo, h_max)

epsr_air = 1.0
epsr_dielectric = 2.0
epsr = mesh.MaterialCF({"air": epsr_air, "dielectric": epsr_dielectric})

FE_order = 3

gf_phi = CapacitorSolver(mesh, FE_order, epsr)

In [24]:
fes_flux = HDiv(mesh, order=FE_order-1)
gf_E = GridFunction(fes_flux)
E = -epsr*grad(gf_phi)
gf_E.Set(E)

In [25]:
error = 1/epsr*(E-gf_E)*(E-gf_E)
Draw(err, mesh)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [26]:
eta2 = Integrate(error, mesh, VOL, element_wise=True)

In [27]:
print(np.array(eta2)[:10], "...")

[2.97086784e-13 4.19945590e-14 2.07842431e-14 1.30466596e-13
 2.25361398e-14 1.28528289e-13 2.48967728e-14 1.25540626e-13
 2.61578744e-14 1.42569658e-13] ...


In [28]:
maxerr = max(eta2)
print ("maxerr = ", maxerr)

maxerr =  0.027694178642234373
